# 📊 Basic ETL Example

Simple ETL pipeline using `mlprep`.

## Scenario
1. Select columns: `id`, `name`, `age`, `city`
2. Filter users 18 years or older
3. Save as Parquet

In [ ]:
!pip install -q "mlprep-rust==0.3.1" pandas pyarrow

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd() / 'outputs'
BASE.mkdir(exist_ok=True)

np.random.seed(42)
n_rows = 100
df = pd.DataFrame({
    'id': range(1, n_rows + 1),
    'name': [f'User_{i}' for i in range(n_rows)],
    'age': np.random.randint(15, 60, size=n_rows),
    'city': np.random.choice(['Tokyo', 'Osaka', 'Nagoya', 'Fukuoka'], size=n_rows),
    'score': np.random.rand(n_rows) * 100
})
df.to_csv('data.csv', index=False)
print('Generated data.csv with 100 rows')
df.head(10)

In [ ]:
pipeline_yaml = '''name: basic_etl
inputs:
  - path: data.csv
    format: csv

steps:
  - type: select
    columns:
      - id
      - name
      - age
      - city
  - type: filter
    condition: "age >= 18"

outputs:
  - path: outputs/output.parquet
    format: parquet
'''

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml)
print(pipeline_yaml)

In [ ]:
!mlprep run pipeline.yaml --streaming --memory-limit 1GB

In [ ]:
import os

if os.path.exists('outputs/output.parquet'):
    output_df = pd.read_parquet('outputs/output.parquet')
    print(f'Output shape: {output_df.shape}')
    print(f'Columns: {output_df.columns.tolist()}')
    print(f'Age range: {output_df["age"].min()} - {output_df["age"].max()}')
    print(f'All ages >= 18: {(output_df["age"] >= 18).all()}')
    output_df.head(10)
else:
    print('Output not found')

In [ ]:
input_df = pd.read_csv('data.csv')
print(f'Input: {len(input_df)} rows, Output: {len(output_df)} rows')
print(f'Filtered out: {len(input_df) - len(output_df)} rows')
print(f'Dropped columns: {set(input_df.columns) - set(output_df.columns)}')